In [1]:
%pip install spacy
%pip install pandas
%pip install scispacy
%pip install sentence_transformers
%pip install bertopic

  Using cached spacy-3.8.16-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (28 kB)
  Using cached spacy_legacy-3.0.12-py2.py3-none-any.whl.metadata (2.8 kB)
  Using cached spacy_loggers-1.0.5-py3-none-any.whl.metadata (23 kB)
  Using cached murmurhash-1.0.15-cp312-cp312-manylinux1_x86_64.manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_5_x86_64.whl.metadata (2.3 kB)
  Using cached cymem-2.0.13-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (9.7 kB)
  Using cached preshed-3.0.13-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (5.2 kB)
  Using cached thinc-8.3.13-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (14 kB)
  Using cached wasabi-1.1.3-py3-none-any.whl.metadata (28 kB)
  Using cached srsly-2.5.3-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (19 kB)
  Using cached catalogue-2.0.10-py3-none-any.whl.metadata (14 kB)
  Using cached weasel-1.0.0-py3-none-any.whl.metadata (4

In [ ]:
%pip install https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_core_sci_md-0.5.4.tar.gz

In [ ]:
%pip install spacy==2.0.18

In [1]:
#imports
import spacy
import scispacy
import timeit
import pandas as pd
import os
from pathlib import Path
from collections import defaultdict
from sentence_transformers import SentenceTransformer
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer
from collections import Counter
import plotly.express as px
import umap
import hdbscan
import torch
import pyarrow.feather as feather
import time

/home/exouser/jupyter-env/lib/python3.12/site-packages/torch/cuda/__init__.py:1112: UserWarning: Can't initialize NVML
  raw_cnt = _raw_device_count_nvml()


In [2]:
#load term parser
nlp = spacy.load("en_core_sci_md")

In [3]:
''' 
wrap this into a class since there are multiple shared attr per instance and there are multiple instances
'''


class Cluster:
    def __init__(self,df):
        device = "cuda" if torch.cuda.is_available() else "cpu"
        print("Using:", device)
        self.df = df
        self.vectorizer = CountVectorizer(lowercase=True)
        self.index = defaultdict(set)
        self.model = SentenceTransformer("paraphrase-MiniLM-L3-v2", device=device)
        self.cl_df = None #filled in once clustering is done
        self.topic_model = None #filled in once clustering is done
        
    
    def _add_spacy_terms(self):
        # Add empty lists for spacy terms
        self.df['spacey_terms'] = self.df.apply(lambda _: [], axis=1)
        for row in self.df.itertuples():
            doc = nlp(row.geo_summary)

            for ent in doc.ents:
                terms = ent.text.split()
                row.spacey_terms.extend(terms)

                for t in terms:
                    self.index[t].add(row.gse)
                    
        # Return flat list of all terms
        return [term for row_terms in self.df['spacey_terms'] for term in row_terms]

    def _update_idx(self):
        analyzer = self.vectorizer.build_analyzer()
        updated = defaultdict(set)
    
        for term, doc_ids in self.index.items():
            tokenized = analyzer(term)
            for token in tokenized:
                updated[token].update(doc_ids)
    
        self.index = updated

    def _add_journals(self):

        self.cl_df['Journals'] = self.cl_df.apply(lambda _: [], axis=1)
        self.cl_df["Representation"] = self.cl_df["Representation"].apply( lambda lst: [x for x in lst if x] )
    
        for row in self.cl_df.itertuples():
            for term in row.Representation:
                if term in self.index:
                    row.Journals.extend(list(self.index[term]))
                else:
                    print(f"term {term} not found in index")
                    
    def cluster_sample(self):
        
      #  print("Adding spacy terms...")
        flat_terms = self._add_spacy_terms()
     #   print("Spacy terms added!")
        
        self.vectorizer.fit(flat_terms)
        self._update_idx()
    
        # Build the embedding + topic model
        
        
        #self.model = self.model.to(device)
       
        umap_model = umap.UMAP(n_neighbors=15, n_components=5, min_dist=0.0)
        hdbscan_model = hdbscan.HDBSCAN(min_samples=10)

        
        topic_model = BERTopic(
            vectorizer_model=self.vectorizer,
            embedding_model=self.model,
            umap_model=umap_model,
            hdbscan_model=hdbscan_model
        )
       # print("Topic modeling initialized!")
        #print(flat_terms)
        topics, probs = topic_model.fit_transform(flat_terms)
       # print("Topic model fitted!")
              
        self.cl_df = topic_model.get_topic_info()
        self._add_journals()

      #  print("Journals added!")
    
        return topic_model, self.cl_df

    def _get_cluster_counts(clusters,cluster_idx): # input cluster_df, returns the number of journals in the clusters
        journals = clusters.iloc[cluster_idx].Journals
        cnts = Counter(journals)
        cnts = dict(cnts)
        return cnts

    def _get_cluster_distribution(clusters,cluster_idx): #returns the journal percent distribution of every cluster
        journals = clusters.iloc[cluster_idx].Journals
        cnts = Counter(journals)
       
        total = len(journals)
    
        percent_dist =  {name: (count / total) * 100 for name, count in cnts.items()}
        return percent_dist

    '''
    returns a dataframe of journal: and its distribution amongst top n clusters
    '''
    def get_top_cluster_distribution(clusters,n=5):
        top_clusters = clusters.head(n)
        distributions = {}
        for idx, row in top_clusters.iterrows():
            distributions[row.Topic] = get_cluster_distribution(clusters,idx)
    
        
        return pd.DataFrame(distributions)

    '''
    returns a dictionary of journal: number of times it appears
    '''
    def get_top_journals(clusters,n=5):
        
        top_clusters = clusters.head(n).iloc[1:] #skip outliers
        overall_cnts = defaultdict(int)
        for idx, row in top_clusters.iterrows():
           # print(f"Processing cluster {row.Topic} at index {idx}")
            journal_counts = get_cluster_counts(clusters,idx)
           # print(f"Journal counts for cluster {row.Topic}: {journal_counts}")
            for journal, count in journal_counts.items():
                #print(f"Adding {count} to overall count for journal {journal}")
                overall_cnts[journal] += count
        
        return overall_cnts

    '''
    returns a dataframe with the journals and what percentage they take up out of all the journals
    '''
    def get_top_journal_distribution(clusters): 
        
        top_cnt_dict = get_top_journals(clusters)
        total = sum(top_cnt_dict.values())
        
        percent_dist = {name: (count / total) * 100 for name, count in top_cnt_dict.items()}
    
        
        return pd.DataFrame(percent_dist.items(),columns=['Journal','Percentage'])

        

In [4]:
def _get_cluster_distribution(clusters,cluster_idx): #returns the journal percent distribution of every cluster
    journals = clusters.iloc[cluster_idx].Journals
    cnts = Counter(journals)
   
    total = len(journals)

    percent_dist =  {name: (count / total) * 100 for name, count in cnts.items()}
    return percent_dist

def _get_cluster_counts(clusters,cluster_idx): #returns the number of journals in the clusters
    journals = clusters.iloc[cluster_idx].Journals
    cnts = Counter(journals)
    cnts = dict(cnts)
    return cnts

'''
returns a dataframe of journal: and its distribution amongst top n clusters
'''
def get_top_cluster_distribution(clusters,n=5):
    top_clusters = clusters.head(n)
    distributions = {}
    for idx, row in top_clusters.iterrows():
        distributions[row.Topic] = _get_cluster_distribution(clusters,idx)

    
    return pd.DataFrame(distributions)

'''
returns a dictionary of journal: number of times it appears
'''
def get_top_journals(clusters,n=5):
    
    top_clusters = clusters.head(n).iloc[1:] #skip outliers
    overall_cnts = defaultdict(int)
    for idx, row in top_clusters.iterrows():
       # print(f"Processing cluster {row.Topic} at index {idx}")
        journal_counts = _get_cluster_counts(clusters,idx)
       # print(f"Journal counts for cluster {row.Topic}: {journal_counts}")
        for journal, count in journal_counts.items():
            #print(f"Adding {count} to overall count for journal {journal}")
            overall_cnts[journal] += count
    
    return overall_cnts

'''
returns a dataframe with the journals and what percentage they take up out of all the journals
'''
def get_top_journal_distribution(clusters): 
    
    top_cnt_dict = get_top_journals(clusters)
    total = sum(top_cnt_dict.values())
    
    percent_dist = {name: (count / total) * 100 for name, count in top_cnt_dict.items()}

    
    return pd.DataFrame(percent_dist.items(),columns=['Journal','Percentage'])

In [5]:
#folder test
start_time = time.time()

folder_path = Path.cwd().parent / "archs4metadata_cohort"
for filename in os.listdir(folder_path): 
    if not filename.endswith(".csv"):
        continue
        
    cohort_name = filename.split('_')[0]
    full_path = os.path.join(folder_path, filename)
    df = pd.read_csv(full_path)
    groups = df.drop(columns=df.columns[0]).groupby("spaceflight")
    grouped = {name: g for name, g in groups}

    for k,v in grouped.items():
        save_name = f"cluster_models/{cohort_name}_{k}.feather"
        
        tc = Cluster(v)
        model, clusters = tc.cluster_sample()
        
        feather.write_feather(clusters, save_name)
        print("Saved:",save_name)

end_time = time.time()    
print(f"Total Time elapsed (seconds):{end_time - start_time}")
    

Using: cpu


Loading weights:   0%|          | 0/55 [00:00<?, ?it/s]

Saved: cluster_models/OSD-464_Ground Control.feather
Using: cpu


Loading weights:   0%|          | 0/55 [00:00<?, ?it/s]

Saved: cluster_models/OSD-464_Space Flight.feather
Using: cpu


...

Saved: cluster_models/OSD-245_Basal Control.feather
Using: cpu


Loading weights:   0%|          | 0/55 [00:00<?, ?it/s]

Saved: cluster_models/OSD-245_Ground Control.feather
Using: cpu


Loading weights:   0%|          | 0/55 [00:00<?, ?it/s]

Saved: cluster_models/OSD-245_Space Flight.feather
Total Time elapsed (seconds):1959.7570552825928


In [11]:
#grab a random set of cohorts 

import random
def sample_cohorts(n=5): #grabs n random cohorts returns a list of them
    folder = Path.cwd().parent / "archs4metadata_cohort"
    filenames = [f.split("_")[0] for f in os.listdir(folder) if f.endswith(".csv")]
    subset = random.sample(filenames, n)
    
    return subset

In [18]:
def get_cohort_data(cohort_name,c1,c2): #retrieves c1 spaceflight cohort and c2 spaceflight cohort
    folder = "cluster_models"
    c1_df = None
    c2_df = None
    for f in os.listdir(folder):
        fl = os.path.join(folder, f)
        if cohort_name and c1 in f:
            c1_df = feather.read_table(fl)
            c1_df = c1_df.to_pandas()
        if cohort_name and c2 in f:
            c2_df = feather.read_table(fl)
            c2_df = c2_df.to_pandas()

    return c1_df,c2_df

In [13]:
sample_cohorts()

['OSD-421', 'OSD-511', 'OSD-613', 'OSD-665', 'OSD-164']

In [19]:
ground,space = get_cohort_data('OSD-421','Ground Control','Space Flight')

In [26]:
print(ground.head(5))



   Topic  Count                                    Name  \
0     -1     93          -1_iss_maintained_rods_control   
1      0     44                 0_alive_study_of_aboard   
2      1     42                        1_spaceflight___   
3      2     39                 2_mice_mammalian_mouse_   
4      3     38  3_retinal_retinitis_vitreoretinopathy_   

                                      Representation  \
0  [iss, maintained, rods, control, decrease, det...   
1  [alive, study, of, aboard, ndpko, compare, cen...   
2                                      [spaceflight]   
3                           [mice, mammalian, mouse]   
4            [retinal, retinitis, vitreoretinopathy]   

                       Representative_Docs  \
0          [control, detected, maintained]   
1                      [of, aboard, alive]   
2  [spaceflight, spaceflight, spaceflight]   
3                       [mice, mice, mice]   
4              [Retinal, retinal, retinal]   

                               

In [27]:
print(space.head(5))

   Topic  Count                                               Name  \
0     -1     32  -1_contacts_downregulation_comprehensive_upreg...   
1      0     46                             0_to_sets_station_days   
2      1     41     1_microgravity_taurine_reducibility_artificial   
3      2     41          2_cysteine_mitochondria_wrapper_mitigated   
4      3     40                                   3_spaceflight___   

                                      Representation  \
0  [contacts, downregulation, comprehensive, upre...   
1  [to, sets, station, days, upstream, secretion,...   
2  [microgravity, taurine, reducibility, artifici...   
3  [cysteine, mitochondria, wrapper, mitigated, c...   
4                                      [spaceflight]   

                                 Representative_Docs  \
0                     [contacts, Contacts, Contacts]   
1                                     [sets, to, to]   
2         [microgravity, microgravity, microgravity]   
3  [wrappER-mitoch

In [77]:
!sudo apt install --reinstall cuda

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
E: Unable to locate package cuda


In [71]:
file_path = Path.cwd().parent / "archs4metadata_cohort/OSD-100_hits.csv"
df = pd.read_csv(file_path)
groups = df.drop(columns=df.columns[0]).groupby("spaceflight")
grouped = {name: g for name, g in groups}

In [72]:
#create a Cluster object and get the topic_model, clusters
tc = Cluster(grouped_dfs[0])

#flat_terms = tc._add_spacy_terms()

Using: cuda


Loading weights:   0%|          | 0/55 [00:00<?, ?it/s]

AcceleratorError: CUDA error: device doesn't have valid Grid license
Search for `cudaErrorDeviceNotLicensed' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
For more detailed error information, run with CUDA_LOG_FILE=stderr


In [32]:
model, clusters = tc.cluster_sample()

In [55]:
file_name = f"cluster_models/"
feather.write_feather(clusters, "cluster_models/clusters.feather")

In [58]:
table = feather.read_table("data.feather")
df = table.to_pandas()

In [51]:
%pip install pyarrow

Note: you may need to restart the kernel to use updated packages.


In [42]:
top_j = get_top_journals(clusters)
jour_dict = get_top_journal_distribution(clusters)

In [44]:
jour_dict

,Journal,Percentage
0,GSE210492,9.615385
1,GSE106591,36.538462
2,GSE143281,9.615385
3,GSE205070,36.538462
4,GSE189618,3.846154
5,GSE124745,3.846154


In [ ]:
#save model to the models folder
topic_model.save("my_bertopic_model", serialization="safetensors")

In [7]:
###OLD###
def _add_spacy_terms(): # adds spacy terms to dataframe, returns list of flat terms of all tokens of all rows
    md_df['spacey_terms'] = md_df.apply(lambda _: [], axis=1)

    for row in md_df.itertuples():
        doc = nlp(row.geo_summary)
        for ent in doc.ents:
            term = ent.text.split()
            row.spacey_terms.extend(term)
            for t in term:
                idx[t].add(row.gse)
            
    
    return [item for lst in md_df["spacey_terms"] for item in lst]


def _update_idx(): #updates the term:journal it came from
    
    analyzer = vectorizer.build_analyzer()
    new_idx = defaultdict(set)

    for term,doc_ids in idx.items():
        tokenized = analyzer(term)
        for token in tokenized:
            new_idx[token].update(doc_ids)

    return new_idx  
    
def _add_journals(cl_df): #adds the list of journals that correspond to the 'Representation' words to the dataframe
     cl_df['journals'] = cl_df.apply(lambda _: [], axis=1)
     cl_df["Representation"] = cl_df["Representation"].apply(lambda lst: [x for x in lst if x])

     for row in cl_df.itertuples():
        
        
        for term in row.Representation:
            
            if term in idx:
               row.journals.extend(list(idx[term]))
            else:
                  print(f"term {term} not found in index")
     
     return cl_df   

'''
#clusters the samples in the provided dataframe. currently not set up to input a dataframe. 
returns the Bertopic model and the cluster dataframe. The cluster dataframe contains the results of the model.get_topic_info
as well as the journals associated with that cluster. 
'''
def cluster_sample(): 
    global idx
    flat_terms = _add_spacy_terms()
    vectorizer.fit(flat_terms)
    idx = _update_idx()
    model = SentenceTransformer('allenai/biomed_roberta_base')
    topic_model = BERTopic(vectorizer_model=vectorizer, embedding_model=model)
    topics,probs = topic_model.fit_transform(flat_terms)
    clusters = _add_journals(topic_model.get_topic_info())   
    return topic_model,clusters

'''
?
'''




In [8]:
df = init_dataframes(Path.cwd().parent / "archs4metadata_cohort/OSD-47_hits.csv")

In [15]:
model.get_topic_info().head(15)

,Topic,Count,Name,Representation,Representative_Docs
0,-1,62,-1_target_physiological_cachexia_metabolisim,"[target, physiological, cachexia, metabolisim,...","[target, target, target]"
1,0,61,0_tissues_seq_mice_noncoding,"[tissues, seq, mice, noncoding, dissected, dis...","[tissues, tissues, tissues]"
2,1,43,1_muscle_muscular__,"[muscle, muscular, , , , , , , , ]","[muscle, muscle, muscle]"
3,2,32,2_genes_gene_activitiy_genetic,"[genes, gene, activitiy, genetic, studies, act...","[genes, genes, genes]"
4,3,30,3_decay_dioxide_resistance_regenerative,"[decay, dioxide, resistance, regenerative, pro...","[sclerosis, oxygen, regenerative]"
5,4,30,4_skeletal___,"[skeletal, , , , , , , , , ]","[skeletal, skeletal, skeletal]"
6,5,24,5_variability_overexpressed_expressed_to,"[variability, overexpressed, expressed, to, la...","[variability, variability, variability]"
7,6,23,6_type_wt_phenotype_histological,"[type, wt, phenotype, histological, control, c...","[type, type, type]"
8,7,21,7_h19ko_dusp27_ampk_pgc,"[h19ko, dusp27, ampk, pgc, muhur, ksrp, ko, 1α...","[DUSP27, AMPK, DUSP27]"
9,8,21,8_hur_formation_rna_production,"[hur, formation, rna, production, carbon, , , ...","[HuR, HuR, HuR]"


In [ ]:
import torch
print(torch.cuda.is_available())

In [27]:
jour_dist

,Journal,Percentage
0,GSE100505,45.714286
1,GSE103202,28.571429
2,GSE134241,25.714286


In [28]:
pd.DataFrame(get_top_cluster_distribution(clusters)) #note: includes outliers as -1 index

,-1,0,1,2,3
GSE103202,18.181818,42.857143,25.0,42.857143,NaN
GSE100505,54.545455,42.857143,50.0,57.142857,40.0
GSE134241,27.272727,14.285714,25.0,NaN,60.0


In [59]:
list(jour_dict.keys())

['GSE100505', 'GSE103202', 'GSE134241']

In [60]:
list(jour_dict.items())

[('GSE100505', 48.64864864864865),
 ('GSE103202', 16.216216216216218),
 ('GSE134241', 35.13513513513514)]

In [61]:
def _plot_dist(dist):
    fig = px.pie(
    names = list(jour_dict.keys()),
    values = list(jour_dict.values()),
   
    title="Percentage Breakdown of Journals",
    )

    fig.show(renderer="browser")


In [62]:
_plot_dist(jour_dist)

In [63]:
def save_cluster_data(clusters, filename="cluster_data.csv"):
    clusters.to_csv(filename, index=False)

In [64]:
save_cluster_data(clusters, filename="COHORT|study=OSD-770|spaceflight=Space_Flight_top10hits_clusters.csv")

In [65]:
def compare_ground_spaceflight(clusters_ground,clusters_spaceflight):
    #see how much the clusters overlap between the two datasets
    #show common clusters, unique clusters, and distribution of journals in each cluster
    

    #create a set of intersection of clusters
    common_clusters = set(clusters_ground["Name"]).intersection(set(clusters_spaceflight["Name"]))
    unique_clusters_ground = set(clusters_ground["Name"]).difference(set(clusters_spaceflight["Name"]))
    unique_clusters_spaceflight = set(clusters_spaceflight["Name"]).difference(set(clusters_ground["Name"]))

    common_journals =   set(clusters_ground["journals"]).intersection(set(clusters_spaceflight["journals"]))
    unique_journals1 = set(clusters_ground["journals"]).difference(set(clusters_spaceflight["journals"]))
    unique_journals2 = set(clusters_spaceflight["journals"]).difference(set(clusters_ground["journals"]))

    #create a dataframe with the information
    comparison_dict ={
        "common_clusters": list(common_clusters),
        "unique_clusters_ground_control": list(unique_clusters_ground),
        "unique_clusters_spaceflight": list(unique_clusters_spaceflight),
        "common_journals": list(common_journals),
        "unique_journals_ground_control": list(unique_journals1),
        "unique_journals_spaceflight": list(unique_journals2)
    }
   
    
                     
                                                



    return comparison_dict

In [66]:
!ls

'COHORT|study=OSD-100|spaceflight=Ground_Control_top10hits_clusters.csv'
'COHORT|study=OSD-100|spaceflight=Space_Flight_top10hits_clusters.csv'
'COHORT|study=OSD-770|spaceflight=Ground Control_top10hits_clusters.csv'
'COHORT|study=OSD-770|spaceflight=Space_Flight_top10hits_clusters.csv'
 cluster_metadata.ipynb
 requirements.txt


In [67]:
import os
print(os.getcwd())

/home/exouser/Desktop/bridge-rna/clustering


In [68]:
ground_file_path = 'COHORT|study=OSD-100|spaceflight=Ground_Control_top10hits_clusters.csv' 
spaceflight_file_path = 'COHORT|study=OSD-100|spaceflight=Space_Flight_top10hits_clusters.csv'



ground_df = pd.read_csv(ground_file_path)
spaceflight_df = pd.read_csv(spaceflight_file_path)

compare_dict = compare_ground_spaceflight(ground_df, spaceflight_df)  